In [1]:
# Cell 1 — Install
!pip install transformers datasets peft bitsandbytes accelerate trl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.1 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import IA3Config, get_peft_model

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [3]:
# Cell 3 — Config
#
# IA3 adds NO new matrices — only three learned scaling vectors per layer.
# Those vectors are element-wise multiplied onto existing activations:
#   • l_k  → rescales Key   outputs  in every attention layer
#   • l_v  → rescales Value outputs  in every attention layer
#   • l_ff → rescales FFN intermediate activations (input to down_proj)
#
# Param math for Llama-3.2-3B-Instruct:
#   hidden_size      = 3072
#   num_layers       = 28
#   num_kv_heads     = 8     (GQA — fewer KV heads than query heads)
#   head_dim         = 128   (hidden_size / num_attention_heads = 3072 / 24)
#   intermediate_size= 8192  (FFN expansion dim)
#
#   l_k  per layer : num_kv_heads × head_dim = 8  × 128  =  1,024
#   l_v  per layer : num_kv_heads × head_dim = 8  × 128  =  1,024
#   l_ff per layer : intermediate_size       = 8,192
#   ─────────────────────────────────────────────────────
#   Per layer total : 1,024 + 1,024 + 8,192            = 10,240
#   All 28 layers   : 28 × 10,240                      = 286,720
#
# Comparison vs same model:
#   Full fine-tuning : ~3,000,000,000  (100%)
#   Prefix Tuning    :     3,440,640   (0.11%)
#   LoRA (r=16)      :    ~13,000,000  (0.43%)
#   IA3              :       286,720   (0.0096%)  ← this notebook

MODEL_NAME     = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 512

In [4]:
# Cell 4 — Load tokenizer and model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Total params      : {total_params:,}")
print(f"Hidden size       : {model.config.hidden_size}")
print(f"Num layers        : {model.config.num_hidden_layers}")
print(f"Num KV heads      : {model.config.num_key_value_heads}")
print(f"Intermediate size : {model.config.intermediate_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Model loaded
Total params      : 1,803,463,680
Hidden size       : 3072
Num layers        : 28
Num KV heads      : 8
Intermediate size : 8192


In [5]:
# Cell 5 — Wrap model with PEFT IA3
#
# KEY DIFFERENCES from Prefix Tuning:
#   1. IA3Config instead of PrefixTuningConfig
#   2. No virtual tokens, no MLP reparametrization — just three vectors per layer
#   3. target_modules: which Linear layers to attach scaling vectors to
#   4. feedforward_modules: subset of target_modules that are FFN layers
#      PEFT uses this to know whether to scale the INPUT (FFN) or OUTPUT (attention)
#      • k_proj, v_proj → scale the OUTPUT  (the computed key / value vectors)
#      • down_proj      → scale the INPUT   (intermediate hidden state before projection)
#
# Why down_proj (not gate_proj or up_proj)?
#   Llama FFN uses SwiGLU:  h = silu(gate_proj(x)) * up_proj(x)
#   The intermediate activation h is the input to down_proj.
#   IA3 gates h — exactly matching the paper's formulation: l_ff ⊙ σ(W1·x)
#   Targeting down_proj is the standard convention in HF PEFT for SwiGLU models.

peft_config = IA3Config(
    task_type          = "CAUSAL_LM",
    target_modules     = ["k_proj", "v_proj", "down_proj"],
    feedforward_modules= ["down_proj"],   # must be a subset of target_modules
    init_ia3_weights   = True,            # initialise all vectors to 1.0 (no-op at start)
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Expected output — roughly:
#   trainable params: 286,720 || all params: 3,212,... || trainable%: 0.009...

trainable params: 286,720 || all params: 3,213,036,544 || trainable%: 0.0089


In [10]:
# Cell 6 — Load, sample, and format MedAlpaca dataset
#
# medalpaca/medical_meadow_medqa contains USMLE-style medical Q&A.
# Columns: 'input' (additional context), 'instruction' (the question),
#          'output' (the answer / explanation)
#
# Unlike bitext, medalpaca has no category column for stratified sampling.
# Instead we shuffle with a fixed seed and take the first N examples.
# N_SAMPLES = 500 is enough to see clear IA3 adaptation in 100 steps
# while keeping Colab T4 runtime under ~10 minutes total.
#
# We merge instruction + input into the user turn.
# If 'input' is empty or whitespace we skip it to avoid a bare newline.

from collections import Counter

N_SAMPLES = 500   # ← adjust upward if you have more time / VRAM

raw_dataset = load_dataset(
    "medalpaca/medical_meadow_medqa",
    split="train",
)

print(f"Full dataset    : {len(raw_dataset):,} examples")

# Shuffle with a fixed seed so results are reproducible, then slice
raw_dataset = raw_dataset.shuffle(seed=42).select(range(N_SAMPLES))

print(f"Sampled dataset : {len(raw_dataset):,} examples")

def format_prompt(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        user_content = instruction.strip()
        if inp and inp.strip():
            user_content = user_content + "\n" + inp.strip()

        text = (
            f"<|begin_of_text|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_content}"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{output.strip()}"
            f"<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.map(format_prompt, batched=True)
print(f"\nFormatted dataset : {len(dataset):,} examples")
print(f"Sample            :")
print(dataset[0]["text"][:300] + "...")

Full dataset    : 10,178 examples
Sampled dataset : 500 examples


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Formatted dataset : 500 examples
Sample            :
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Please answer with one of the option in the bracket
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies...


In [16]:
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Please answer with one of the option in the bracket
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:? 
{'A': 'Pallor, cyanosis, and erythema of the hands', 'B': 'Calcium deposits on digits', 'C': 'Blanching vascular abnormalities', 'D': 'Hypercoagulable state', 'E': 'Heartburn and regurgitation'},<|eot_id|><|start_header_id|>assistant<|end_header_id|>

D: Hypercoagulable state<|eot_id|>


In [11]:
# Cell 7 — Tokenize with label masking (loss only on assistant response)
#
# KEY IMPROVEMENT over naive tokenization:
#   Plain labels = input_ids means the model trains on the user prompt too.
#   That wastes capacity — we only want to teach the model HOW TO ANSWER,
#   not to memorise the questions.
#
# Masking strategy:
#   1. Find the assistant header token sequence in each example.
#   2. Set labels to -100 for every token UP TO and INCLUDING the header.
#   3. Set labels to -100 for all padding tokens.
#   Trainer ignores positions where label == -100 when computing loss.
#
# This is identical in spirit to the bitext notebook's label masking —
# the only difference is the medalpaca examples tend to be longer
# (USMLE explanations vs one-liner customer support answers),
# so the trainable-token ratio per example is higher here.

ASSISTANT_HEADER     = "<|start_header_id|>assistant<|end_header_id|>\n\n"
assistant_header_ids = tokenizer.encode(ASSISTANT_HEADER, add_special_tokens=False)
header_len           = len(assistant_header_ids)

def tokenize(examples):
    tokens = tokenizer(
        examples["text"],
        truncation = True,
        max_length = MAX_SEQ_LENGTH,
        padding    = "max_length",
    )
    labels = []
    for ids in tokens["input_ids"]:
        label = list(ids)

        # Scan for the assistant header and mask everything before it
        found = False
        for i in range(len(label) - header_len + 1):
            if label[i : i + header_len] == assistant_header_ids:
                label[: i + header_len] = [-100] * (i + header_len)
                found = True
                break

        # If header not found (rare truncation edge-case) mask the whole example
        if not found:
            label = [-100] * len(label)

        # Mask padding tokens
        pad_id = tokenizer.pad_token_id
        for i in range(len(label)):
            if ids[i] == pad_id:
                label[i] = -100

        labels.append(label)

    tokens["labels"] = labels
    return tokens

tokenized_dataset = dataset.map(
    tokenize,
    batched        = True,
    remove_columns = dataset.column_names,
    num_proc       = 2,
)

# Sanity check: how many tokens per example are actually trained on?
import numpy as np

sample_labels  = tokenized_dataset[0]["labels"]
n_train_tokens = sum(1 for t in sample_labels if t != -100)
n_masked       = sum(1 for t in sample_labels if t == -100)

print(f"Tokenized dataset : {len(tokenized_dataset):,} examples")
print(f"Columns           : {tokenized_dataset.column_names}")
print(f"Sample label check: {n_train_tokens} trainable tokens, {n_masked} masked (of {len(sample_labels)})")
print()
print("If trainable tokens = 0 the assistant header was not found — check your tokenizer.")

Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenized dataset : 500 examples
Columns           : ['input_ids', 'attention_mask', 'labels']
Sample label check: 8 trainable tokens, 504 masked (of 512)

If trainable tokens = 0 the assistant header was not found — check your tokenizer.


In [12]:
# Cell 8 — Train
#
# Learning rate is higher than prefix tuning (3e-3 vs 5e-3).
# IA3 has far fewer trainable params (287K vs 3.4M) so individual
# vectors need a slightly larger update signal to move meaningfully.
# The vectors start at 1.0 (identity), so even a modest LR drives
# fast convergence — 100 steps is enough to see clear adaptation.
#
# No weight_decay on the IA3 vectors themselves (they're tiny scalars;
# regularising them toward zero would undo the learned rescaling).

training_args = TrainingArguments(
    output_dir                  = "outputs",
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    learning_rate               = 3e-4,
    warmup_steps                = 30,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    logging_steps               = 50,
    seed                        = 42,
    report_to                   = "none",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss
50,2.285954
100,1.995534
150,1.893921
200,1.807900
250,1.740385


TrainOutput(global_step=250, training_loss=1.9447386474609376, metrics={'train_runtime': 1730.4949, 'train_samples_per_second': 0.289, 'train_steps_per_second': 0.144, 'total_flos': 4330036396032000.0, 'train_loss': 1.9447386474609376, 'epoch': 1.0})

In [15]:
# Cell 9 — Inference after training
model.eval()

compute_dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

test_questions = [
    "What is the first-line treatment for uncomplicated community-acquired pneumonia in a healthy adult?",
    "A 45-year-old patient presents with chest pain radiating to the left arm and diaphoresis. What is the most likely diagnosis and immediate management?",
    "Explain the mechanism of action of metformin in type 2 diabetes.",
    "What are the classic signs of Cushing's syndrome and which diagnostic test confirms it?",
]

print("=" * 60)
print("AFTER TRAINING — IA3 Tuned Model Responses (Medical)")
print("=" * 60)

for question in test_questions:
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}"
        f"<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        with torch.autocast("cuda", dtype=compute_dtype):   # ← fixes the dtype mismatch
            outputs = model.generate(
                **inputs,
                max_new_tokens = 200,
                temperature    = 0.7,
                do_sample      = True,
                pad_token_id   = tokenizer.eos_token_id,
            )

    prompt_len = inputs["input_ids"].shape[1]
    response   = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

AFTER TRAINING — IA3 Tuned Model Responses (Medical)

Q: What is the first-line treatment for uncomplicated community-acquired pneumonia in a healthy adult?
A: The first-line treatment for uncomplicated community-acquired pneumonia (CAP) in a healthy adult is typically a combination of antibiotics and supportive care.

According to the Infectious Diseases Society of America (IDSA) guidelines, the preferred first-line treatment for CAP in adults is:

* A macrolide antibiotic (e.g., azithromycin or clarithromycin) or a fluoroquinolone antibiotic (e.g., levofloxacin or moxifloxacin) for 7-10 days.

The IDSA guidelines also recommend that the choice of antibiotic should be guided by the following factors:

* The severity of the pneumonia
* The patient's age and underlying health conditions
* The presence of comorbidities (e.g., heart disease, diabetes)
* The patient's ability to take oral antibiotics
* The potential for antibiotic resistance

Supportive care may include:

* Rest and hydrat